# 01 — Audio Streaming: WAV Files & Real-time PCM

**Audience**: Developers who completed notebook 00 and want to work with real audio  
**Duration**: ~25 minutes  
**Goal**: Learn the audio format requirements, generate and stream WAV/PCM data to Gemini, and simulate real mic streaming

---

## Introduction: Audio Formats for the Live API

Getting audio formats right is the most common source of confusion when using the Live API. Here is everything you need to know up front:

### PCM16 — the baseline audio format

**PCM** (Pulse-Code Modulation) is the simplest possible digital audio representation: a sequence of integer samples, one per time step. **PCM16** means each sample is a signed 16-bit integer (`int16`), range −32 768 to +32 767.

There are **no headers, no compression, no codec** — just raw numbers.

```
┌────────┬────────┬────────┬────────┬─────┐
│ s[0]   │ s[1]   │ s[2]   │ s[3]   │ ... │  ← 2 bytes each, little-endian
│ int16  │ int16  │ int16  │ int16  │     │
└────────┴────────┴────────┴────────┴─────┘
```

### Gemini's requirements

| Direction | Format | Sample rate | Channels |
|---|---|---|---|
| **Input** (you → Gemini) | PCM16 | **16 000 Hz** | Mono |
| **Output** (Gemini → you) | PCM16 | **24 000 Hz** | Mono |

### Why different rates?

- **16 kHz input** — sufficient for speech intelligibility (human voice bandwidth is ~300–3400 Hz; Nyquist: 8 kHz is enough, 16 kHz gives headroom)
- **24 kHz output** — higher fidelity for synthesised speech sounds more natural

### WAV vs PCM

A **WAV file** is just a PCM stream with an **44-byte RIFF header** prepended that describes the format (sample rate, bit depth, channels). To send audio to Gemini, you strip the header and send only the raw PCM bytes.

## Step 1: Install Dependencies

In [1]:
!pip install google-genai nest_asyncio numpy --quiet

## Step 2: Setup & Imports

In [2]:
import nest_asyncio; nest_asyncio.apply()

import asyncio
import io
import os
import struct
import wave

import numpy as np
import IPython.display as ipd

from google import genai
from google.genai import types

from dotenv import load_dotenv
load_dotenv()  # loads GEMINI_API_KEY from .env

API_KEY = os.environ.get("GEMINI_API_KEY", "")
MODEL   = "gemini-3.1-flash-live-preview"
client  = genai.Client(api_key=API_KEY)

# ── Audio constants ────────────────────────────────────────────────────────────
INPUT_RATE  = 16_000   # Hz — required by Gemini for input audio
OUTPUT_RATE = 24_000   # Hz — Gemini always outputs at this rate
CHUNK_SIZE  = 512      # samples per chunk (≈ 32 ms at 16kHz) — typical mic buffer size

print("✓ Setup complete")
print(f"  Input rate  : {INPUT_RATE} Hz")
print(f"  Output rate : {OUTPUT_RATE} Hz")
print(f"  Chunk size  : {CHUNK_SIZE} samples ({CHUNK_SIZE/INPUT_RATE*1000:.0f} ms)")

✓ Setup complete
  Input rate  : 16000 Hz
  Output rate : 24000 Hz
  Chunk size  : 512 samples (32 ms)


## Audio Format Helpers

These utility functions handle all the PCM/WAV conversions needed in this notebook.

In [3]:
# ══════════════════════════════════════════════════════════════════════════════
# AUDIO FORMAT HELPERS
# ══════════════════════════════════════════════════════════════════════════════

def make_pcm(text_hint: str = "", duration: float = 2.0, rate: int = 16000) -> bytes:
    """
    Generate a sine-wave tone as raw PCM16 bytes.
    Use 'low' in text_hint for 220 Hz, otherwise 440 Hz.
    """
    freq    = 220 if "low" in text_hint else 440
    t       = np.linspace(0, duration, int(rate * duration), endpoint=False)
    samples = (np.sin(2 * np.pi * freq * t) * 0.3 * 32767).astype(np.int16)
    return samples.tobytes()


def play_pcm(raw_bytes: bytes, rate: int = 24000) -> ipd.Audio:
    """Wrap raw PCM16 bytes in an IPython Audio widget."""
    arr = np.frombuffer(raw_bytes, dtype=np.int16).astype(np.float32) / 32768.0
    return ipd.Audio(arr, rate=rate, autoplay=False)


def pcm_to_wav_bytes(pcm: bytes, rate: int = 16000, channels: int = 1) -> bytes:
    """
    Wrap raw PCM16 bytes in a WAV container (RIFF header + data).
    The wave module handles the 44-byte RIFF header for us.

    Args:
        pcm      : raw PCM16 bytes (little-endian int16)
        rate     : sample rate in Hz
        channels : 1 = mono, 2 = stereo

    Returns:
        Complete WAV file as bytes (header + PCM data)
    """
    buf = io.BytesIO()
    with wave.open(buf, "wb") as wf:
        wf.setnchannels(channels)      # mono
        wf.setsampwidth(2)             # 2 bytes = 16-bit
        wf.setframerate(rate)
        wf.writeframes(pcm)            # write the raw samples
    return buf.getvalue()              # includes RIFF header


def wav_bytes_to_pcm(wav_bytes: bytes) -> tuple[bytes, int]:
    """
    Extract raw PCM16 bytes from a WAV file (strips the header).

    Returns:
        (pcm_bytes, sample_rate)
    """
    buf = io.BytesIO(wav_bytes)
    with wave.open(buf, "rb") as wf:
        rate  = wf.getframerate()
        nch   = wf.getnchannels()
        sw    = wf.getsampwidth()
        n_frames = wf.getnframes()
        raw   = wf.readframes(n_frames)
        print(f"  WAV info: {rate} Hz, {nch} ch, {sw*8}-bit, {n_frames} frames")
    return raw, rate


def resample_pcm(pcm: bytes, from_rate: int, to_rate: int) -> bytes:
    """
    Simple linear resampling of PCM16 bytes.
    For production use, prefer scipy.signal.resample or librosa.
    """
    samples    = np.frombuffer(pcm, dtype=np.int16).astype(np.float32)
    n_out      = int(len(samples) * to_rate / from_rate)
    resampled  = np.interp(
        np.linspace(0, len(samples), n_out),
        np.arange(len(samples)),
        samples
    ).astype(np.int16)
    return resampled.tobytes()


def split_pcm_into_chunks(pcm: bytes, chunk_samples: int = 512) -> list[bytes]:
    """
    Split a PCM16 byte string into fixed-size chunks.
    Each chunk is chunk_samples * 2 bytes (2 bytes per int16 sample).
    The final chunk is zero-padded if needed.
    """
    chunk_bytes = chunk_samples * 2          # 2 bytes per int16
    chunks = []
    for i in range(0, len(pcm), chunk_bytes):
        chunk = pcm[i : i + chunk_bytes]
        if len(chunk) < chunk_bytes:         # pad last chunk
            chunk = chunk + b"\x00" * (chunk_bytes - len(chunk))
        chunks.append(chunk)
    return chunks


print("✓ Audio helpers defined")
print("  Functions: make_pcm, play_pcm, pcm_to_wav_bytes, wav_bytes_to_pcm,")
print("             resample_pcm, split_pcm_into_chunks")

✓ Audio helpers defined
  Functions: make_pcm, play_pcm, pcm_to_wav_bytes, wav_bytes_to_pcm,
             resample_pcm, split_pcm_into_chunks


---
## Demo 1: Multi-Tone Test Audio → Gemini Audio Response

We generate a **chord** — four simultaneous sine waves at different frequencies — to create a richer test signal than a single tone. This is useful for verifying that Gemini's input pipeline handles more complex audio.

Steps:
1. Generate a four-tone chord (C major: 261, 329, 392, 523 Hz)
2. Visualise the waveform in the notebook
3. Convert to PCM16 at 16 kHz
4. Send to Gemini, collect audio response

In [4]:
# ── Generate a multi-tone chord ────────────────────────────────────────────────
DURATION = 2.0       # seconds
RATE     = INPUT_RATE  # 16000 Hz

t = np.linspace(0, DURATION, int(RATE * DURATION), endpoint=False)

# C major chord: C4, E4, G4, C5
freqs = [261.63, 329.63, 392.00, 523.25]
chord = np.zeros_like(t)

for freq in freqs:
    # Each tone at 1/4 amplitude so the combined signal doesn't clip
    chord += np.sin(2 * np.pi * freq * t) * 0.25

# Apply a simple fade-in and fade-out (10 ms each) to avoid clicks
fade_samples = int(RATE * 0.01)   # 10 ms
fade_in  = np.linspace(0, 1, fade_samples)
fade_out = np.linspace(1, 0, fade_samples)
chord[:fade_samples]  *= fade_in
chord[-fade_samples:] *= fade_out

# Convert to PCM16 bytes
chord_int16 = (chord * 32767).astype(np.int16)
chord_pcm   = chord_int16.tobytes()

print(f"Generated C-major chord:")
print(f"  Frequencies : {freqs} Hz")
print(f"  Duration    : {DURATION}s")
print(f"  Sample rate : {RATE} Hz")
print(f"  PCM16 bytes : {len(chord_pcm):,}")
print(f"  Samples     : {len(chord_int16):,}")

# Display the waveform
print("\nPlayback (listen to the chord):")
ipd.display(play_pcm(chord_pcm, rate=RATE))

Generated C-major chord:
  Frequencies : [261.63, 329.63, 392.0, 523.25] Hz
  Duration    : 2.0s
  Sample rate : 16000 Hz
  PCM16 bytes : 64,000
  Samples     : 32,000

Playback (listen to the chord):


In [5]:
async def demo_chord_to_gemini():
    """
    Send the multi-tone chord to Gemini and receive a spoken response.
    """
    config = types.LiveConnectConfig(
        response_modalities=["AUDIO"],
        system_instruction=types.Content(parts=[types.Part(text=(
            "You are a helpful music teacher. "
            "The user will send you audio. "
            "If it is a musical tone or chord, identify it and say something brief about it. "
            "If it sounds like noise or silence, say so politely."
        ))]),
        realtime_input_config=types.RealtimeInputConfig(
            automatic_activity_detection=types.AutomaticActivityDetection(disabled=True)
        ),
    )

    audio_blob = types.Blob(data=chord_pcm, mime_type="audio/pcm;rate=16000")

    response_chunks = []
    async with client.aio.live.connect(model=MODEL, config=config) as session:
        await session.send_realtime_input(activity_start=types.ActivityStart())
        _chunk_bytes = 512 * 2  # 512 samples × 2 bytes per int16
        for _i in range(0, len(chord_pcm), _chunk_bytes):
            _chunk = chord_pcm[_i:_i+_chunk_bytes]
            await session.send_realtime_input(
                audio=types.Blob(data=_chunk, mime_type="audio/pcm;rate=16000")
            )
        await session.send_realtime_input(activity_end=types.ActivityEnd())

        async for resp in session.receive():
            if resp.data:
                response_chunks.append(resp.data)
            if resp.server_content and resp.server_content.turn_complete:
                break
            if resp.go_away:
                break

    return b"".join(response_chunks)


print("Sending chord to Gemini ...")
resp_audio = asyncio.run(demo_chord_to_gemini())
print(f"Response: {len(resp_audio):,} bytes ({(len(resp_audio)//2)/24000:.2f}s)")
print("Gemini's response:")
play_pcm(resp_audio, rate=OUTPUT_RATE)

Sending chord to Gemini ...
Response: 336,482 bytes (7.01s)
Gemini's response:


---
## Demo 2: WAV File Workflow

In production you often load audio from `.wav` files. This demo shows the complete workflow:

1. **Create** a WAV file in memory (simulating reading from disk)
2. **Inspect** its headers
3. **Extract** the raw PCM bytes (strip the header)
4. **Resample** if the WAV is not at 16 kHz
5. **Send** to Gemini

In [6]:
# ── Step 1: Create a WAV file in memory ────────────────────────────────────────
# We build a WAV at 44100 Hz (CD quality) to show the resampling step.

WAV_RATE = 44100   # CD quality — intentionally NOT 16000 to show resampling
duration = 1.5     # seconds

t_wav    = np.linspace(0, duration, int(WAV_RATE * duration), endpoint=False)
# A simple two-tone signal: 300 Hz + 600 Hz
signal   = (
    np.sin(2 * np.pi * 300 * t_wav) * 0.4 +
    np.sin(2 * np.pi * 600 * t_wav) * 0.2
).astype(np.float32)

pcm_44k  = (signal * 32767).astype(np.int16).tobytes()

# Wrap in a WAV container using the wave module
wav_bytes = pcm_to_wav_bytes(pcm_44k, rate=WAV_RATE, channels=1)

print(f"Created in-memory WAV file:")
print(f"  Total bytes       : {len(wav_bytes):,}")
print(f"  Header bytes      : 44 (standard RIFF WAV header)")
print(f"  PCM data bytes    : {len(wav_bytes) - 44:,}")
print(f"  Sample rate       : {WAV_RATE} Hz")
print(f"  Duration          : {duration}s")

Created in-memory WAV file:
  Total bytes       : 132,344
  Header bytes      : 44 (standard RIFF WAV header)
  PCM data bytes    : 132,300
  Sample rate       : 44100 Hz
  Duration          : 1.5s


In [7]:
# ── Step 2: Inspect WAV headers manually ──────────────────────────────────────
# The first 44 bytes of a WAV file follow the RIFF specification.
# Let's read a few key fields to understand the structure.

def inspect_wav_header(wav_bytes: bytes) -> None:
    """Print key fields from a WAV RIFF header."""
    # RIFF header layout (little-endian):
    # Offset  Size  Field
    # 0       4     ChunkID      = b"RIFF"
    # 4       4     ChunkSize    = file size - 8
    # 8       4     Format       = b"WAVE"
    # 12      4     Subchunk1ID  = b"fmt "
    # 16      4     Subchunk1Size= 16 for PCM
    # 20      2     AudioFormat  = 1 for PCM
    # 22      2     NumChannels
    # 24      4     SampleRate
    # 28      4     ByteRate     = SampleRate * NumChannels * BitsPerSample/8
    # 32      2     BlockAlign   = NumChannels * BitsPerSample/8
    # 34      2     BitsPerSample
    # 36      4     Subchunk2ID  = b"data"
    # 40      4     Subchunk2Size= number of PCM data bytes
    h = wav_bytes[:44]
    chunk_id        = h[0:4]
    chunk_size      = struct.unpack_from("<I", h, 4)[0]
    fmt_id          = h[8:12]
    audio_format    = struct.unpack_from("<H", h, 20)[0]
    num_channels    = struct.unpack_from("<H", h, 22)[0]
    sample_rate     = struct.unpack_from("<I", h, 24)[0]
    byte_rate       = struct.unpack_from("<I", h, 28)[0]
    bits_per_sample = struct.unpack_from("<H", h, 34)[0]
    data_size       = struct.unpack_from("<I", h, 40)[0]

    print(f"WAV Header Inspection:")
    print(f"  ChunkID       : {chunk_id}  (must be b'RIFF')")
    print(f"  Format        : {fmt_id}  (must be b'WAVE')")
    print(f"  AudioFormat   : {audio_format}  (1 = uncompressed PCM)")
    print(f"  NumChannels   : {num_channels}  (1=mono, 2=stereo)")
    print(f"  SampleRate    : {sample_rate:,} Hz")
    print(f"  ByteRate      : {byte_rate:,} bytes/sec")
    print(f"  BitsPerSample : {bits_per_sample} bits")
    print(f"  DataSize      : {data_size:,} bytes  ({data_size/(sample_rate*2):.3f}s of audio)")

inspect_wav_header(wav_bytes)

WAV Header Inspection:
  ChunkID       : b'RIFF'  (must be b'RIFF')
  Format        : b'WAVE'  (must be b'WAVE')
  AudioFormat   : 1  (1 = uncompressed PCM)
  NumChannels   : 1  (1=mono, 2=stereo)
  SampleRate    : 44,100 Hz
  ByteRate      : 88,200 bytes/sec
  BitsPerSample : 16 bits
  DataSize      : 132,300 bytes  (1.500s of audio)


In [8]:
# ── Step 3: Extract PCM and resample to 16 kHz ────────────────────────────────

print("Extracting PCM from WAV:")
pcm_extracted, extracted_rate = wav_bytes_to_pcm(wav_bytes)

print(f"  Extracted {len(pcm_extracted):,} bytes at {extracted_rate} Hz")

if extracted_rate != INPUT_RATE:
    print(f"  Resampling {extracted_rate} Hz → {INPUT_RATE} Hz ...")
    pcm_for_gemini = resample_pcm(pcm_extracted, from_rate=extracted_rate, to_rate=INPUT_RATE)
    print(f"  Resampled to {len(pcm_for_gemini):,} bytes at {INPUT_RATE} Hz")
else:
    pcm_for_gemini = pcm_extracted
    print(f"  Already at {INPUT_RATE} Hz, no resampling needed")

print("\nOriginal 44.1 kHz audio:")
ipd.display(play_pcm(pcm_extracted, rate=extracted_rate))

print("\nResampled 16 kHz audio (sounds the same, lower quality):")
ipd.display(play_pcm(pcm_for_gemini, rate=INPUT_RATE))

Extracting PCM from WAV:
  WAV info: 44100 Hz, 1 ch, 16-bit, 66150 frames
  Extracted 132,300 bytes at 44100 Hz
  Resampling 44100 Hz → 16000 Hz ...
  Resampled to 48,000 bytes at 16000 Hz

Original 44.1 kHz audio:



Resampled 16 kHz audio (sounds the same, lower quality):


In [9]:
async def demo_wav_to_gemini(pcm_bytes: bytes) -> bytes:
    """
    Send resampled WAV audio to Gemini and return the spoken response.
    """
    config = types.LiveConnectConfig(
        response_modalities=["AUDIO"],
        system_instruction=types.Content(parts=[types.Part(text=(
            "You received audio from a user. "
            "If it sounds like a tone, say: 'I received your WAV audio successfully.' "
            "Then briefly describe what you heard."
        ))]),
        realtime_input_config=types.RealtimeInputConfig(
            automatic_activity_detection=types.AutomaticActivityDetection(disabled=True)
        ),
    )

    chunks = []
    async with client.aio.live.connect(model=MODEL, config=config) as session:
        await session.send_realtime_input(activity_start=types.ActivityStart())
        _chunk_bytes = 512 * 2  # 512 samples × 2 bytes per int16
        for _i in range(0, len(pcm_bytes), _chunk_bytes):
            _chunk = pcm_bytes[_i:_i+_chunk_bytes]
            await session.send_realtime_input(
                audio=types.Blob(data=_chunk, mime_type="audio/pcm;rate=16000")
            )
        await session.send_realtime_input(activity_end=types.ActivityEnd())
        async for resp in session.receive():
            if resp.data:
                chunks.append(resp.data)
            if resp.server_content and resp.server_content.turn_complete:
                break
            if resp.go_away:
                break

    return b"".join(chunks)


print("Sending resampled WAV audio to Gemini ...")
wav_response = asyncio.run(demo_wav_to_gemini(pcm_for_gemini))
print(f"Response received: {len(wav_response):,} bytes")
play_pcm(wav_response, rate=OUTPUT_RATE)

Sending resampled WAV audio to Gemini ...


ConnectionClosedError: received 1007 (invalid frame payload data) Precondition check failed.; then sent 1007 (invalid frame payload data) Precondition check failed.

---
## Demo 3: Chunked Streaming (Simulated Microphone)

Real microphone input doesn't arrive all at once — it comes in small buffers (typically 512–4096 samples at a time). This demo simulates that pattern:

1. Split a 4-second audio clip into 512-sample chunks (~32 ms each)
2. Send one chunk at a time with a small delay between chunks
3. Receive the streaming audio response

This is exactly how a production voice assistant would work:

```
Mic callback →  [512 samples] → send_realtime_input(audio=...) → repeat
                [512 samples] ↗
                [512 samples] ↗
                ...
```

In [ ]:
# ── Prepare a 4-second audio clip to stream as chunks ──────────────────────────
# We use a melody: 4 notes, 1 second each
STREAM_DURATION = 4.0  # seconds total
notes = [
    (440.0, 1.0),   # A4 — 1 second
    (493.9, 1.0),   # B4 — 1 second
    (523.3, 1.0),   # C5 — 1 second
    (587.3, 1.0),   # D5 — 1 second
]

melody_parts = []
for freq, dur in notes:
    t_note  = np.linspace(0, dur, int(INPUT_RATE * dur), endpoint=False)
    note    = (np.sin(2 * np.pi * freq * t_note) * 0.3 * 32767).astype(np.int16)
    melody_parts.append(note)

melody_samples = np.concatenate(melody_parts)
melody_pcm     = melody_samples.tobytes()

# Split into 512-sample chunks
chunks = split_pcm_into_chunks(melody_pcm, chunk_samples=CHUNK_SIZE)

print(f"Melody prepared:")
print(f"  Notes       : A4, B4, C5, D5 (1s each)")
print(f"  Duration    : {STREAM_DURATION}s")
print(f"  Total bytes : {len(melody_pcm):,}")
print(f"  Chunks      : {len(chunks)} × {CHUNK_SIZE} samples ({CHUNK_SIZE/INPUT_RATE*1000:.0f}ms each)")

print("\nPreview melody:")
ipd.display(play_pcm(melody_pcm, rate=INPUT_RATE))

In [ ]:
async def demo_chunked_stream(chunks: list[bytes]) -> bytes:
    """
    Stream audio to Gemini one small chunk at a time, simulating a microphone.

    The delay between chunks (CHUNK_SIZE/INPUT_RATE seconds) matches the
    real-time playback speed — as if each chunk arrived from a live mic buffer.
    """
    config = types.LiveConnectConfig(
        response_modalities=["AUDIO"],
        system_instruction=types.Content(parts=[types.Part(text=(
            "You are a music assistant. The user is streaming audio to you in chunks. "
            "When the audio is complete, briefly describe what you heard, "
            "then say 'Streaming demo complete.'"
        ))]),
        realtime_input_config=types.RealtimeInputConfig(
            automatic_activity_detection=types.AutomaticActivityDetection(disabled=True)
        ),
    )

    chunk_delay = CHUNK_SIZE / INPUT_RATE   # real-time delay between chunks (≈32ms)
    response_audio = []

    async with client.aio.live.connect(model=MODEL, config=config) as session:
        print(f"[Session open — streaming {len(chunks)} chunks with {chunk_delay*1000:.0f}ms delay each]")

        # ── Stream chunks to Gemini ────────────────────────────────────────────
        await session.send_realtime_input(activity_start=types.ActivityStart())
        for i, chunk in enumerate(chunks):
            blob = types.Blob(data=chunk, mime_type="audio/pcm;rate=16000")
            await session.send_realtime_input(audio=blob)

            # Simulate real-time pacing — wait as long as the chunk duration
            # In a real app this delay comes from the OS audio callback timing
            await asyncio.sleep(chunk_delay)

            if (i + 1) % 25 == 0:   # progress every 25 chunks (~0.8s)
                elapsed = (i + 1) * chunk_delay
                print(f"  Streamed {i+1}/{len(chunks)} chunks ({elapsed:.1f}s elapsed)")

        await session.send_realtime_input(activity_end=types.ActivityEnd())
        print(f"  All {len(chunks)} chunks sent — waiting for response")

        # ── Collect response ───────────────────────────────────────────────────
        async for resp in session.receive():
            if resp.data:
                response_audio.append(resp.data)
            if resp.server_content and resp.server_content.turn_complete:
                print("[turn complete]")
                break
            if resp.go_away:
                print("[go_away]")
                break

    print("[Session closed]")
    return b"".join(response_audio)


print("── Demo 3: Chunked Streaming ──\n")
stream_response = asyncio.run(demo_chunked_stream(chunks))

dur = (len(stream_response) // 2) / OUTPUT_RATE
print(f"\nResponse: {len(stream_response):,} bytes ({dur:.2f}s)")
play_pcm(stream_response, rate=OUTPUT_RATE)

---
## Audio Format Conversion: Complete Reference

Here is a summary of all the conversion paths you might need in a real application.

In [ ]:
# ── Conversion paths reference ──────────────────────────────────────────────────

# ┌─────────────────────────┐     ┌───────────────────────────────────────────┐
# │  Your audio source      │     │  What Gemini needs                        │
# └────────────┬────────────┘     └───────────────────────────────────────────┘
#              │
#   ┌──────────▼──────────────────────────────────────────────────────────────┐
#   │  Source: WAV file (44.1 kHz, 16-bit, mono)                              │
#   │  Steps:                                                                  │
#   │    1. pcm, rate = wav_bytes_to_pcm(open("file.wav", "rb").read())        │
#   │    2. if rate != 16000: pcm = resample_pcm(pcm, rate, 16000)            │
#   │    3. send: types.Blob(data=pcm, mime_type="audio/pcm;rate=16000")       │
#   └─────────────────────────────────────────────────────────────────────────┘
#
#   ┌──────────────────────────────────────────────────────────────────────────┐
#   │  Source: Microphone (PyAudio, sounddevice, etc.)                         │
#   │  Steps:                                                                  │
#   │    1. Configure mic at 16000 Hz, 16-bit, mono                            │
#   │    2. Each callback: send raw frames as Blob(mime_type="audio/pcm;rate=16000") │
#   │    3. No conversion needed if mic is already at 16kHz                    │
#   └─────────────────────────────────────────────────────────────────────────┘
#
#   ┌──────────────────────────────────────────────────────────────────────────┐
#   │  Output: Gemini audio → save as WAV file                                 │
#   │  Steps:                                                                  │
#   │    1. Collect all resp.data chunks                                       │
#   │    2. wav_bytes = pcm_to_wav_bytes(b"".join(chunks), rate=24000)         │
#   │    3. open("response.wav", "wb").write(wav_bytes)                        │
#   └─────────────────────────────────────────────────────────────────────────┘

# ── Demo: Save a response as a WAV file ────────────────────────────────────────
if stream_response:
    wav_out = pcm_to_wav_bytes(stream_response, rate=OUTPUT_RATE)
    output_path = "/tmp/gemini_response.wav"
    with open(output_path, "wb") as f:
        f.write(wav_out)
    print(f"Saved response to {output_path} ({len(wav_out):,} bytes)")
    print("Verifying saved file:")
    inspect_wav_header(wav_out)

---
## Key Takeaways

1. **PCM16 is raw audio** — no headers, no codec — just signed 16-bit integers, 2 bytes per sample.

2. **Gemini requires 16 kHz input**. Always resample before sending if your source is at a different rate.

3. **Gemini outputs at 24 kHz**. Set `rate=24000` when playing or saving responses.

4. **WAV files = RIFF header (44 bytes) + PCM data**. Use the `wave` module to read/write; strip the header before sending to Gemini.

5. **Chunked streaming** (512-sample buffers with real-time delays) is how production voice apps work — stream each mic callback directly without buffering the whole utterance.

6. **`types.Blob(data=pcm_bytes, mime_type="audio/pcm;rate=16000")`** is the correct wrapper for audio input.

7. **Fade in/out** your synthesised audio to avoid clicks at the start and end of clips.

---
**Next notebook →** `02_transcription.ipynb` — Input and output transcription for accessibility, logging, and search